# Sofware per la gestione di un negozio vegano

In [1]:
# Import moduli e package
import csv
import os
# import sys
import shutil
from tempfile import NamedTemporaryFile
from prettytable import from_csv

# Variables:
INVENTORY = "inventario_prodotti.csv"
SALES = "sales.csv"

In [2]:
def files_existence_check():
    
    """Function to check the existence of the "inventory_products.csv" and "sales.csv" files: if the files do not exist they are created"""
    
    inventory_existence = os.path.exists(INVENTORY)
    sales_existence = os.path.exists(SALES)
    if inventory_existence is False:
        print("Creazione del magazzino")
        fields = ["Product", "Amount", "SellingPrice", "PurchasePrice", "DeltaPrice"]

        with open(INVENTORY, "w", newline='') as inventory:
            writer = csv.DictWriter(inventory, fieldnames = fields)
            writer.writeheader()
            
    if sales_existence is False:
        print("Creazione del registro delle vendite")
        fields = ["Product", "Amount", "SellingPrice", "DeltaPrice"]

        with open(SALES, "w", newline='') as sales:
            writer = csv.DictWriter(sales, fieldnames = fields)
            writer.writeheader()


def product_presence_check(product):
    
    """Function for checking whether the product is in stock"""
    
    with open(INVENTORY, encoding = "utf-8", mode = "r") as check_prod_existence:
        reader = csv.reader(check_prod_existence)
        for row in reader:
            if product == row[0]:
                return True
        return False


def new_product_registration(product, amount, purchase_price, selling_price):
    
    """Function to register a new product not in stock"""
    
    with open(INVENTORY, encoding = "utf-8", mode = "a", newline='') as new_record:
        csv_writer = csv.writer(new_record)
        csv_writer.writerow([product, amount, selling_price, purchase_price, delta_price])        
        print(f"Aggiunte {amount} unità del prodotto {product}")


def quantity_update(product, amount):
    
    """
    Function for updating the quantity of a product already in stock
    For this purpose, a temporary file has been created with the same characteristics as the original one. After that, the content of the original file is copied to the temporary file, updating the quantity of the product.
    At the end, the shutil.move function proceeds to overwrite the original file with the temp file, renaming it with the name of the original one.
    """
    
    TEMPFILE = NamedTemporaryFile(mode="w", encoding = "utf-8", delete=False, newline='')
    fields = ["Product", "Amount", "SellingPrice", "PurchasePrice", "DeltaPrice"]

    with open(INVENTORY, encoding = "utf-8", mode = "r", newline='') as csvfile, TEMPFILE:
        reader = csv.DictReader(csvfile, fieldnames=fields)
        writer = csv.DictWriter(TEMPFILE, fieldnames=fields)
        for row in reader:
            if row["Product"] == str(product):
                print("Aggiorno la riga relativa al prodotto", row["Product"])
                row["Amount"] = int(row["Amount"]) + amount # se è una vendita il valore di amount sarà negativo
                row["SellingPrice"], row["PurchasePrice"], row["DeltaPrice"] = selling_price, purchase_price, delta_price
            row = {"Product": row["Product"], "Amount": row["Amount"], "SellingPrice": row["SellingPrice"], "PurchasePrice": row["PurchasePrice"], "DeltaPrice": row["DeltaPrice"]}
            writer.writerow(row)

    shutil.move(TEMPFILE.name, INVENTORY)


def show_products():
    
    """Function to show the products in stock"""
  
    with open(INVENTORY, encoding = "utf-8", mode = "r") as csv_file:
        mytable = from_csv(csv_file)
        print(mytable)     


def amount_presence_check(product, amount):
    
    """Function for checking the presence of the quantity of product in stock"""
    
    with open(INVENTORY, encoding = "utf-8", mode = "r") as csv_file:
        csv_reader = csv.DictReader(csv_file)
        for row in csv_reader:
            if row["Product"] == str(product):
                if int(row["Amount"]) < amount:
                    return False
                else:
                    return True 


def sales_registration(product, amount):
    
    """Function for recording the sale in sales.csv"""
    
    TEMPFILE = NamedTemporaryFile(mode="w", encoding = "utf-8", delete=False, newline='')
    
    with open(INVENTORY, encoding = "utf-8", mode = "r", newline='') as inventory, TEMPFILE, open(SALES, encoding = "utf-8", mode = "a+", newline='') as new_sell:
        
        inventory_fields = ["Product", "Amount", "SellingPrice", "PurchasePrice", "DeltaPrice"]
        sales_fields = ["Product", "Amount", "SellingPrice", "DeltaPrice"]
        
        inventory_reader = csv.DictReader(inventory, fieldnames=inventory_fields)
        temfile_writer = csv.DictWriter(TEMPFILE, fieldnames=inventory_fields)
        sale_writer = csv.DictWriter(new_sell, fieldnames=sales_fields)       
    
        for row in inventory_reader:
            if row["Product"] == product:
                print(f"aggiorno la riga relativa al prodotto {product} in inventario_prodotti.csv")
                
                selling_price, delta_price = row["SellingPrice"], row["DeltaPrice"]
                row["Amount"] = int(row["Amount"]) - amount
                
            row = {"Product": row["Product"], "Amount": row["Amount"], "SellingPrice": row["SellingPrice"], "PurchasePrice": row["PurchasePrice"], "DeltaPrice": row["DeltaPrice"]}            
            temfile_writer.writerow(row)
         
        row["Product"], row['Amount'], row["SellingPrice"], row["DeltaPrice"] = product, amount, selling_price, delta_price
        row = {"Product": row["Product"], "Amount": row["Amount"], "SellingPrice": row["SellingPrice"], "DeltaPrice": row["DeltaPrice"]}
        sale_writer.writerow(row)           
        print(f"Venduti: {amount} unità di {product}")    
        
        
    shutil.move(TEMPFILE.name, INVENTORY)


def show_profits(product, amount, selling_price, delta_price):
    
    """Function for calculating the total net and gross profit"""
    
    total_gross_profit = 0
    total_net_profit = 0
    
    with open(SALES, encoding = "utf-8", mode = "r") as csv_file:
        csv_reader = csv.DictReader(csv_file)
        for row in csv_reader:
            total_gross_profit = round(total_gross_profit + (float(row["SellingPrice"])*int(row["Amount"])),2)
            total_net_profit = round(total_net_profit + (float(row["DeltaPrice"])*int(row["Amount"])),2)
        print(f"Profitti_lordi: {total_gross_profit}€, Profitti_netti: {total_net_profit}€")


def int_check(num_var):
    
    """Function for checking that Amount variable is a positive int"""
    
    while True:
        try:
            if int(num_var) <= 0:
                print("Il valore inserito non è un numero intero positivo")
                num_var = input("Inserire nuovamente il valore per continuare, altrimenti digita [no] per chiudere l'operazione: ").lower().rstrip().lstrip()
                if num_var == "no":
                    return False, None
                continue
            return True, int(num_var)
            break
        except ValueError:
            print("Il valore inserito deve essere un numero intero positivo")
            num_var = input("Inserire nuovamente il valore per continuare, altrimenti digita [no] per chiudere l'operazione: ").lower().rstrip().lstrip()
            if num_var == "no":
                return False, None
            continue


def float_check(num_var):
    
    """Function for checking that Selling_price and Purchase_price variables are positive floats"""
    
    while True:
        try:
            mum_var = float(num_var)
            if float(num_var) <= 0:
                print("Il valore inserito non è un numero a virgola mobile")
                num_var = input("Inserire nuovamente il valore per continuare, altrimenti digita [no] per chiudere l'operazione: ").lower().rstrip().lstrip()                
                if num_var == "no":
                    return False, None
                continue
            return True, float(num_var)
            break
        except ValueError:
            print("Il valore inserito deve essere un numero a virgola mobile")
            num_var = input("Inserire nuovamente il valore per continuare, altrimenti digita [no] per chiudere l'operazione: ").lower().rstrip().lstrip()
            if num_var == "no":
                return False, None
            continue

In [5]:
#Software commands creation:
print("Benvenuto!")
files_existence_check()
print(f"Inserisci un comando presente nel seguente elenco:\n"
                        f"[aggiungi] : aggiungi un prodotto al magazzino\n"
                        f"[elenca] : elenca i prodotti e le quantità presenti in magazzino\n"
                        f"[vendita] : registra una vendita effettuata\n"
                        f"[profitti] : mostra i profitti totali lordi e netti\n"
                        f"[aiuto] : mostra i possibili comandi\n"
                        f"[chiudi] : esci dal programma\n"
                        )

while True:
    user_command = input().lower().rstrip().lstrip()
    if user_command == "aggiungi":   
        product = input("Prodotto acquistato: ").lower().rstrip().lstrip()
        amount = input("Quantità acquistata: ")
        check_amount = int_check(amount)
        if check_amount[0] == True:
            amount = check_amount[1]
        else: 
            print("chiusura dell'operazione")
            break
        purchase_price = input("Prezzo unitario di acquisto (e.g: 15.99): ")
        check_purchase_price = float_check(purchase_price)
        if check_purchase_price[0] == True:
            purchase_price = check_purchase_price[1]
        else: 
            print("chiusura dell'operazione")
            break            
        selling_price = input("prezzo unitario di vendita (e.g: 19.99): ")
        check_selling_price = float_check(selling_price)
        if check_selling_price[0] == True:
            selling_price = check_selling_price[1]
        else: 
            print("chiusura dell'operazione")
            break            
        delta_price = round(selling_price - purchase_price, 2)
        if product_presence_check(product) == True:
            quantity_update(product, amount)
        else:       
            new_product_registration(product, amount, purchase_price, selling_price)
    elif user_command == "elenca":
        show_products()
    elif user_command == "vendita":
        product = input("Prodotto venduto: ").lower().rstrip().lstrip()
        if product_presence_check(product) == False:
            print("Il prodotto non è presente in magazzino")
        else:
            amount = input("Quantità venduta: ")
            check_amount = int_check(amount)
            if check_amount[0] == True:
                amount = check_amount[1]
            else: 
                print("chiusura dell'operazione")
                break        
            if amount_presence_check(product, amount) == True:
                    sales_registration(product, amount)
                    while input("Aggiungere un altro prodotto (si/no)? ").lower().rstrip().lstrip() == "si":                      
                        product = input("Prodotto venduto: ").lower().rstrip().lstrip()
                        if product_presence_check(product) == False:
                            print("Il prodotto non è presente in magazzino")
                        else:
                            amount = input("Quantità venduta: ")
                            check_amount = int_check(amount)
                            if check_amount[0] == True:
                                amount = check_amount[1]
                            else:
                                print("chiusura dell'operazione")
                                break
                            if amount_presence_check(product, amount) == True:
                                sales_registration(product, amount)
                            else:
                                print("La quantità richiesta non è disponibile. Impossibile completare la vendita")
            else:
                print("La quantità richiesta non è disponibile. Impossibile completare la vendita")
    elif user_command == "profitti":
        show_profits(product, amount, selling_price, delta_price) 
    elif user_command == "aiuto":
        print(f"Inserisci un comando presente nel seguente elenco:\n"
                           f"[aggiungi] : aggiungi un prodotto al magazzino\n"
                           f"[elenca] : elenca i prodotti e le quantità presenti in magazzino\n"
                           f"[vendita] : registra una vendita effettuata\n"
                           f"[profitti] : mostra i profitti totali lordi e netti\n"
                           f"[aiuto] : mostra i possibili comandi\n"
                           f"[chiudi] : esci dal programma\n"
                          )
    elif user_command == "chiudi":
        print("Hai scelto il comando di chiusura. Arrividerci")
        break
    else:
        print("Comando non valido. I comandi disponibili sono i seguenti:\n"
                           f"[aggiungi] : aggiungi un prodotto al magazzino\n"
                           f"[elenca] : elenca i prodotti e le quantità presenti in magazzino\n"
                           f"[vendita] : registra una vendita effettuata\n"
                           f"[profitti] : mostra i profitti totali lordi e netti\n"
                           f"[aiuto] : mostra i possibili comandi\n"
                           f"[chiudi] : esci dal programma\n")

Benvenuto!
Inserisci un comando presente nel seguente elenco:
[aggiungi] : aggiungi un prodotto al magazzino
[elenca] : elenca i prodotti e le quantità presenti in magazzino
[vendita] : registra una vendita effettuata
[profitti] : mostra i profitti totali lordi e netti
[aiuto] : mostra i possibili comandi
[chiudi] : esci dal programma

+---------------+--------+--------------+---------------+------------+
|    Product    | Amount | SellingPrice | PurchasePrice | DeltaPrice |
+---------------+--------+--------------+---------------+------------+
| latte di soia |   30   |     1.4      |      0.8      |    0.6     |
|      tofu     |   6    |     4.19     |      2.2      |    1.99    |
|     seitan    |   5    |     5.49     |      3.0      |    2.49    |
|     caffè     |   5    |     7.99     |      4.99     |    3.0     |
+---------------+--------+--------------+---------------+------------+
Aggiunte 3 unità del prodotto formaggio
Comando non valido. I comandi disponibili sono i segue